In [ ]:
#Connecting to Local/ Remote Cloud Data

# Spark JARs: We pull in Sedona and Hadoop AWS packages so we can handle spatial ops and remote files.
# S3 endpoint: We’re connecting to a custom HTTPS-based S3-compatible endpoint, not AWS. Sedona doesn’t 
# care—it just needs to know how to talk to it.
# Memory tuning: Even locally, bumping driver/executor memory can help when working with larger datasets.

In [3]:
from sedona.spark import SedonaContext

config = (
    SedonaContext.builder()
    
    # Connect to the JAR (Java ARchive) packages
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.sedona:sedona-spark-3.5_2.12:1.6.1",
            "org.datasyslab:geotools-wrapper:1.7.0-28.5",
            "org.apache.hadoop:hadoop-aws:3.3.2"
        ])
    )
    .config("spark.jars.repositories", "https://artifacts.unidata.ucar.edu/repository/unidata-all")
    
    # Connect to remote data on Source Cooperative - you will need to sign up for an account and get your access and secret keys
    
    .config("spark.hadoop.fs.s3a.endpoint", "https://data.source.coop") 
    .config("spark.hadoop.fs.s3a.access.key", "SOURCE_COOP_S3_ACCESS_KEY")
    .config("spark.hadoop.fs.s3a.secret.key", "SOURCE_COOP_S3_SECRET_KEY")
    
    # Enable S3 access
    
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    
    # You can add this if you want to use public S3 data, for now we use local  
    
    # .config("spark.hadoop.fs.s3a.aws.credentials.provider", 
    #         "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider") 
    
    .config("spark.executor.memory", "12G")
    .config("spark.driver.memory", "12G")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

sedona = SedonaContext.create(config)
sedona.sparkContext.setLogLevel("ERROR")

# Allowing Spark to fetch resources (jars, data) directly from HTTPS endpoints 
sedona.conf.set("fs.https.impl", "org.apache.hadoop.fs.http.HttpsFileSystem")

https://artifacts.unidata.ucar.edu/repository/unidata-all added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.sedona#sedona-spark-3.5_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fe498710-e2ac-416c-a2a5-627ce8e78c89;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-3.5_2.12;1.6.1 in central
	found org.apache.sedona#sedona-common;1.6.1 in central
	found org.apache.commons#commons-math3;3.6.1 in central
	found org.locationtech.jts#jts-core;1.19.0 in central
	found org.wololo#jts2geojson;0.16.1 in central
	found org.locationtech.spatial4j#spatial4j;0.8 in central
	found com.google.geometry#s2-geometry;2

In [5]:
# Creating simple object row inside a list

In [6]:
from pyspark.sql import Row

In [7]:
data = [
    Row(id = 1, name = "Point A", lat = 40.7128, lon = -74.0060),
    Row(id = 2, name = "Point B", lat = 34.0522, lon = -118.2437),
    Row(id = 3, name = "Point C", lat = 37.7749, lon = -122.4194)
]  

In [8]:
df = sedona.createDataFrame(data)
df.show()

+---+-------+-------+---------+
| id|   name|    lat|      lon|
+---+-------+-------+---------+
|  1|Point A|40.7128|  -74.006|
|  2|Point B|34.0522|-118.2437|
|  3|Point C|37.7749|-122.4194|
+---+-------+-------+---------+



In [9]:
from pyspark.sql.functions import expr

In [13]:
# Adding geom column
df_geom = df.withColumn("geom"
                        , expr("ST_Point(cast(lon as Decimal(24, 20)), cast(lat as Decimal(24, 20)))")
                       )

In [15]:
df_geom.show(truncate = False)

+---+-------+-------+---------+-------------------------+
|id |name   |lat    |lon      |geom                     |
+---+-------+-------+---------+-------------------------+
|1  |Point A|40.7128|-74.006  |POINT (-74.006 40.7128)  |
|2  |Point B|34.0522|-118.2437|POINT (-118.2437 34.0522)|
|3  |Point C|37.7749|-122.4194|POINT (-122.4194 37.7749)|
+---+-------+-------+---------+-------------------------+



In [16]:
sql = """
SELECT ST_AreaSpheroid(
    ST_GeomFromWKT('Polygon ((34 35, 28 30, 25 34, 34 35))')
) as result
"""

sedona.sql(sql).show(truncate = False)

+---------------------+
|result               |
+---------------------+
|2.0182485081176245E11|
+---------------------+



In [18]:
df_geom.createOrReplaceTempView("points")

In [19]:
sedona. sql("""
SELECT id, name, ST_AsText(geom) AS wkt
FROM points
WHERE ST_Within(geom, ST_GeomFromText('POLYGON((-120 20, -60 20, -60 50, -120 50, -120 20))'))
""") .show(truncate = False)

+---+-------+-------------------------+
|id |name   |wkt                      |
+---+-------+-------------------------+
|1  |Point A|POINT (-74.006 40.7128)  |
|2  |Point B|POINT (-118.2437 34.0522)|
+---+-------+-------------------------+



In [ ]:
## Reading vector data

In [22]:
geojson_path = '../data/Neighborhood_Map_Atlas_Neighborhoods.geojson'
geojson_df = sedona.read.format("geojson").load(geojson_path)

Py4JJavaError: An error occurred while calling o122.load.
: java.util.ServiceConfigurationError: org.apache.spark.sql.sources.DataSourceRegister: org.apache.spark.sql.execution.datasources.parquet.GeoParquetFileFormat Unable to get public no-arg constructor
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:586)
	at java.base/java.util.ServiceLoader.getConstructor(ServiceLoader.java:679)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.hasNextService(ServiceLoader.java:1240)
	at java.base/java.util.ServiceLoader$LazyClassPathLookupIterator.hasNext(ServiceLoader.java:1273)
	at java.base/java.util.ServiceLoader$2.hasNext(ServiceLoader.java:1309)
	at java.base/java.util.ServiceLoader$3.hasNext(ServiceLoader.java:1393)
	at scala.collection.convert.JavaCollectionWrappers$JIteratorWrapper.hasNext(JavaCollectionWrappers.scala:46)
	at scala.collection.StrictOptimizedIterableOps.filterImpl(StrictOptimizedIterableOps.scala:225)
	at scala.collection.StrictOptimizedIter

In [ ]:
geojason_df.printSchema()

In [ ]:
#unnesting sedona datafram
from pyspark.sql.functions import expr

In [ ]:
geojson_df = (
    sedona.read.format("geojson").option("multiLine", "true").load(geojson_path)
    .selectExpr("explode(features) as features")
    .select ("features.*")
    .withColumn("L_HOOD", expr("properties['L_HOOD']") )
    .withColumn("OBJECTID", expr("properties['OBJECTID']") )
    .withColumn("S_HOOD", expr("properties['S_HOOD']") )
    .withColumn("S_HOOD_ALT_NAMES", expr("properties ['S_HOOD_ALT_NAMES']"))
    .drop("properties").drop("type")
)

In [ ]:
geojason_df.printSchema()

In [ ]:
geojason_df.show()

In [ ]:
geojason_df.count()

In [ ]:
df = (
    sedona.read.format("geopackage")
    .option("showMetadata", "true")
    .load("data/parks.gpkg")
)

df.show()

In [ ]:
shapefile_path = 'data/parks'
shapefile_df = sedona.read.format("shapefile").load(shapefile_path)

In [ ]:
shapefile_df.show()

In [ ]:
# CSV - https://www.kaggle.com/datasets/andykrause/kingcountysales
csv_path = 'data/kingco_sales.csv'
csv_df = sedona.read.format("csv").load(csv_path)
csv_df.show(3)

In [ ]:
# Read CSV with header and inferSchema
csv_df = sedona.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(csv_path)

csv_df.show(3)

In [ ]:
from pyspark.sql. functions import col 

In [ ]:
csv_df = csv_df.selectExpr(
    "*", "ST_Point(longitude, latitude) as geometry"
)

In [ ]:
csv_df.show(3)

In [ ]:
csv_df.count()

In [ ]:
# GeoParquet
geoparquet_path = 'data/Seattle_Transportation_Plan_Bicycle_Element_829016546957421557.parquet'
geoparquet_df = sedona.read.format("geoparquet").load(geoparquet_path)
geoparquet_df.show(3)

In [ ]:
# Check geomtry
geoparquet_df.selectExpr('ST_SRID(geometry)').show(1)

In [ ]:
geoparquet_df.selectExpr('''ST_Transform(geometry, 'EPSG:2926', 'EPSG:4326') ''').show(1)

In [ ]:
#GeoParquet
geoparquet_schools_path = 'data/Seattle_Public_Schools_Sites_2023-2024.parquet'
geoparquet_schools_df = sedona.read.format("geoparquet").load(geoparquet_schools_path)
geoparquet_schools_df.show(3)

In [ ]:
## Reading raster data

In [ ]:
netcdf_file = "data/pdsi_current_PRISM.nc"
netcdf_df = sedona. read. format("binaryFile").load(netcdf_file)
netcdf_df.show()

In [ ]:
import pyspark. sql. functions as f

In [ ]:
netcdf_df.repartition(8)

In [ ]:
netcdf_df = netcdf_df.withColumn("raster", f.expr("RS_FromNetCDF(content, 'data', 'longitude', 'latitude') "))

In [ ]:
netcdf_df.show()

In [ ]:
# https://radiantearth.github.io/stac-browser/#/external/earth-search.aws.element84.com/v1/collections/sentinel-2-12a/items/S2A_10TET_20250423_0_L2A?.asset

In [ ]:
# https://browser.apex.esa.int/external/eoresults.esa.int/stac/collections/ESA_WORLDCOVER_10M_2021_V2/items/ESA_WorldCover_10m_2021_v200_N45W123?

In [ ]:
### GEOTIFF

In [ ]:
raster_df = sedona.read.format("binaryFile").load("data/ESA_WorldCover_10m_2021_v200_N45W123_Map.tif")

In [ ]:
raster_df = raster_df.withColumn("raster", f.expr("RS_FromGeoTiff(content)"))
# raster_df.show()

In [ ]:
raster_df.repartition(8)

In [ ]:
#Setting tile size for/thinking of retrieval efficiency
raster_df.selectExpr("RS_TileExplode(raster, 256, 256) as (x, y, raster)").show()

In [ ]:
#Also check free data on Earth on AWS

In [ ]:
#Vector and raster viz

In [ ]:
from sedona.spark import SedonaUtils

In [ ]:
# Convert raster to base64 image for inline visualization
htmlDF = raster_df.selectExpr("RS_AsImage(raster, 500) as raster_image")

# Display in notebook (also check Databricks)
SedonaUtils.display_image(htmlDF)

In [ ]:
from sedona.spark import SedonaKepler

map = SedonaKepler.create_map()
SedonaKepler.add_df(map, geoparquet_schools_df, name = "Seattle Schools")
map

In [ ]:
#Vector functions

In [ ]:
from sedona.ql.st_functions import ST_Buffer, ST_Transform, ST_ConvexHull
from pyspark.sql.functions import col, lit

In [ ]:
# Creating buffers of 100m around schools, reprojecting so to use meters
buffered_df = geoparquet_schools_df.select(
                ST_Buffer(
                    ST_Transform(col("geometry"), lit("EPSG:4326"), lit("EPSG:26910"))
                    , 100
                        ).alias("buffer")
              )

buffered_df.show(3)

In [ ]:
# Same shit but using SQL
schools_df = geoparquet_schools_df.selectExpr("*", "ST_Transform(geometry, 'EPSG:4326', 'EPSG:26910') as geom").drop("geometry").withColumnRenamed("geom", "geometry")

In [ ]:
#Transform sedona databasew to view table so to use seemesly SQL
shapefile_df.createOrReplaceTempView('parks')

In [ ]:
centroid = sedona.sql('''select st_centroid(geometry) from parks''')
centroid.show(3)

In [ ]:
area = sedona.sql('''select st_area(geometry) from parks''')
area.show(3)

In [ ]:
geoparquet_df.create0rReplaceTempView('bikes')

In [ ]:
length = sedona. sql('''select st_length(geometry) from bikes''')
length.show(3)

In [ ]:
hull_df = shapefile_df.select(
    "geometry",
    ST_ConvexHull("geometry").alias("convex_hull")
    )

hull_df.show(3)

In [ ]:
#Distance parks to Space Needle - 122.3494692, 47.6205426

In [ ]:
from pyspark.sql.functions import lit

dist_df = sedona. sql('''
select ST_Distance(geometry,
        ST_Transform(
            ST_SetSRID(
                ST_Point(-122.3494692, 47.6205426), 4326), 'EPSG:26910')) as distance
        from parks
                    order by 1
        ''')

dist_df.show(3)

In [ ]:
#Returns numrical array of h3 polygons hierarchical ids
h3_df = sedona.sql('''
    select ST_H3CellIDs(geometry, 8, true) as h3
    from parks
    limit 3
    ''')

h3_df.show()

In [ ]:
#Distance considering curvature and elevation, can also use a 4th dimensions for telemetry
sedona.sql("""
    SELECT ST_3DDistance(
        ST_GeomFromText('POINT Z(000)')
       ,ST_GeomFromText('POINT Z (3 4 12)')
    ) AS dist_3d
""").show()

In [ ]:
#Spatial predicates

In [ ]:
csv_df.createOrReplaceTempView('homes')
geojson_df.createOrReplaceTempView('neighborhoods')

In [ ]:
geojson_df.show(3)

In [ ]:
sedona.sql('''
    select h.sale_id, st_contains(n.geometry, h.geometry) in_ballard
    from homes h
    join neighborhoods n on st_contains(n.geometry, h.geometry)
    where n.S_HOOD = 'Ballard'
    ''').show(3)

In [ ]:
sedona.sql('''
    select h.sale_id
    from homes h
    join neighborhoods n on st_dwithin(
        st_transform(n.geometry, 'EPSG:4326', 'EPSG:26910')
        ,st_transform(h.geometry, 'EPSG:4326', 'EPSG:26910')
    ,500)
where n.S_HOOD = 'Ballard'
''').count ()

In [ ]:
sedona.sql(''' SELECT ST_Contains(
     ST_GeomFromText('POLYGON((0 0,0 10,10 10,10 0,0 0))')
    ,ST_GeomFromText('POINT(5 5)')
    ) AS contains
''').show()

In [ ]:
sedona. sql(''' SELECT ST_Contains(
    ST_GeomFromText('POLYGON((0 0, 0 10, 10 10,10 0,0 0))')
    ,ST_GeomFromText('POINT(0 0)')
    ) AS contains
''').show ()

In [ ]:
# Write data

In [ ]:
# Write a single GeoJSON file (e.g., for download or web map use)
geojson_output_path = "parks.geojson"

shapefile_df.coalesce(1).write \
.mode ("overwrite") \
. format ("geojson") \
.save(geojson_output_path)

In [ ]:
# Repartition to 10 files for parallel writes
csv_df.repartition(10).write \
.mode ("overwrite") \
.format ("geoparquet") \
.save("data/output_geoparquet_10_parts")

In [ ]:
csv_df.write \
.mode ("overwrite") \
.format("geoparquet") \
.partitionBy("cty") \
.save("data/output_geoparquet_partitioned")

In [ ]:
# Writing to cloud bucket

# output_df.write \
# .mode("overwrite") \
# .format("geoparquet") \
# .save("s3a://your-bucket-name/path/to/output/")

In [ ]:
# NN Join

In [ ]:
geoparquet_schools_df. repartition(8)

In [ ]:
geoparquet_schools_df.create0rReplaceTempView('schools')

In [ ]:
# Finding 3 nearest bike rutes to schools
nearest = sedona.sql('''
    select schools. school_name, parks.NAME
    from schools
    join parks on ST_KNN(
         schools.geometry
        ,parks.geometry
        ,3
        true
)
''')

In [ ]:
nearest.show(10)

In [ ]:
# Raster functions

In [ ]:
sample_area = 'POLYGON((-122.367948 47.642361, -122.361745 47.642361, -122.361745 47.638922, -122.367948 47.638922, -122.367948 47.642361))

In [ ]:
raster_df = raster_df.selectExpr("RS_TileExplode(raster, 256, 256) as (x, y, raster)")

In [ ]:
raster_df.create0rReplaceTempView('landcover')

In [ ]:
# Using 1st band, return list of centroid poitn geometry, pixel value and raster x and y coordinates for eacht pixel
centroids = sedona. sql (f'''
    select RS_PixelAsCentroids(raster, 1)
    from landcover
    where rs_intersects(raster, st_geomfromtext('{sample_area}'))
''')

In [ ]:
centroids.show(3)

In [ ]:
raster_df.createOrReplaceTempView('landcover')

In [ ]:
raster_tiles = raster_df.selectExpr("RS_TileExplode(raster, 256, 256) as (x, y, raster)")

In [ ]:
raster_tiles.create0rReplaceTempView('landcover_tiles')

In [ ]:
raster_tiles.show(truncate = False)

In [ ]:
stats = sedona. sql('''SELECT RS_SummaryStatsAll(raster) AS stats
FROM landcover_tiles
where x = 0 and y = 0''')

stats.show(truncate = False)

In [ ]:
metadata = sedona. sql('''SELECT RS_MetaData(raster) AS stats
FROM landcover_tiles
where x = 0 and y = 0''')

metadata.show(truncate=False)

In [ ]:
# Raster map algebra

In [ ]:
# # https://radiantearth.github.io/stac-browser/#/external/earth-search.aws.element84.com/v1/collections/sentinel-2-l

In [ ]:
from pyspark.sql import functions as f

In [ ]:
#Dataset for red-band
red_df = sedona.read.format("binaryFile") \
.load("data/B04.tif")

red_df = red_df.withColumn("raster", f.expr("RS_FromGeoTiff(content)"))

In [ ]:
#Dataset of near infra-red band
nir_df = sedona. read. format("binaryFile") \
.load("data/B08.tif")

nir_df = nir_df.withColumn("raster", f.expr("RS_FromGeoTiff(content)"))

In [ ]:
red_df.createOrReplaceTempView('red')
nir_df.createOrReplaceTempView('nir')

In [ ]:
#Create a 2band raster
union = sedona. sql('''
select RS_Union(red.raster, nir. raster) as raster from red, nir
''')

In [ ]:
union.createOrReplaceTempView('union')

In [ ]:
# NDVI = (NIR - Red) / (NIR + Red)

ndvi = sedona. sql ('''
SELECT
    RS_MapAlgebra(
        raster
        ,'D'
        ,'out = (rast[1] - rast[0])/(rast[1] + rast[0]);'
    ) AS ndvi
FROM union
''')

In [ ]:
ndvi.show()

In [ ]:
from sedona.spark import SedonaUtils

# Convert raster to base64 image for inline visualization
htmlDF = ndvi.selectExpr("RS_AsImage(ndvi, 500) as raster_image")

# Display 
SedonaUtils.display_image(htmlDF)

In [ ]:
# Raster write

In [ ]:
# NOT to run locally
# ndvi.withColumn("raster_binary", expr("RS_AsGeoTiff(ndvi)"))\
# .write. format("raster")\
# .option("rasterField""raster_binary")\
# .option("pathField", "path")\
# .option("fileExtension", ".tiff")\
# .mode("overwrite")\
# .save("my_raster_file")

In [ ]:
# Zonal statistics

In [ ]:
ndvi.createOrReplaceTempView('ndvi')

In [ ]:
park = 'POLYGON((-122.3607955072 47.6460058027, -122.3582580324 47.6460058027, -122. 3582580324 47.6432482444, -122.360 

In [ ]:
parks_ndvi = sedona. sql (f'''
select
RS_ZonalStats(ndvi.ndvi, st_transform(
    st_geomfromtext('{park}'), 'epsg:4326', 'epsg:32610'
    ), 1, 'avg', false, false) as avg_ndvi
from ndvi
''')

In [ ]:
ndvi = ndvi.repartition(8)

In [ ]:
parks_ndvi.show(3)

In [ ]:
# No run all the code in the cloud using Whereobots

In [ ]:
## Import data, pre set ingest credentials and set-up AWS account

In [ ]:
aws s3 sync Users/mXXXXXXXXXXXXX/sedona-tutorial/data s3://wbts-wbc-ymm1bun8sj/jf3gkm4ile/data/
customer-besg0oop07pktb/tutorial_data/ -- recursive

In [ ]:
## Start notebook in whereobots